In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import time
import psutil as pst
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import scvi
import anndata as ad
import pandas as pd
import torch
import seaborn as sns
from scipy.stats import median_abs_deviation

In [ ]:
MERSCOPE_DATADIR = '../data/merscope_hcc1'
RESOLVI_RESULTS_DIR = '../data/resolvi_results'

datasets = ['tx05', 'tx10', 'tx15', 'tx20', 'tx25', 'tx30']

In [ ]:
def load_data(cell_metadata_filepath, tx_metadata_filepath):
    transcript_metadata = pd.read_csv(tx_metadata_filepath).rename({'Unnamed: 0': 'molecule_id'}, axis=1)
    cell_metadata = pd.read_csv(cell_metadata_filepath, index_col=0)
    counts_df = pd.pivot_table(transcript_metadata, index='cell_id', columns='gene', aggfunc='count', values='molecule_id')
    return counts_df, cell_metadata

def create_adata_object(counts_df, cell_metadata):
    adata = sc.AnnData(counts_df)
    cell_metadata = cell_metadata.reindex(adata.obs.index)
    adata.obs = cell_metadata

    adata.layers['counts'] = adata.X
    adata.layers['counts'] = np.nan_to_num(adata.layers['counts'], nan=0)
    adata.obsm['X_spatial'] = adata.obs[['center_x', 'center_y']].to_numpy()
    return adata

def memory_usage():
    """Get current memory usage in MB"""
    process = pst.Process(os.getpid())
    mem_info = process.memory_info()
    return mem_info.rss / (1024 * 1024)  # Return memory in MB

def setup_resolvi(adata, accelerator='cpu'):
    scvi.external.RESOLVI.setup_anndata(adata, labels_key="cell_type", layer="counts")
    resolvi_model = scvi.external.RESOLVI(adata, semisupervised=True)
    resolvi_model.train(max_epochs=100, accelerator=accelerator)
    return resolvi_model

def resolvi_predict(adata, resolvi_model):
    adata.obsm["resolvi_celltypes"] = resolvi_model.predict(adata, num_samples=3, soft=True)
    adata.obs["resolvi_predicted"] = adata.obsm["resolvi_celltypes"].idxmax(axis=1)

def get_latent_representation_and_neighbors(adata, resolvi_model):
    adata.obsm["X_resolVI"] = resolvi_model.get_latent_representation(adata)
    adata.obsm["X_resolVI"] = resolvi_model.get_latent_representation(adata)
    sc.pp.neighbors(adata, use_rep="X_resolVI")
    sc.tl.umap(adata)

def sample_posterior_and_add_to_adata(adata, resolvi_model):
    # Sample posterior for corrected rates
    samples_corr = resolvi_model.sample_posterior(
        model=resolvi_model.module.model_corrected,
        return_sites=["px_rate"],
        summary_fun={"post_sample_q50": np.median},
        num_samples=10,
       summary_frequency=100
    )
    samples_corr = pd.DataFrame(samples_corr).T

    # Sample posterior for mixture proportions
    samples = resolvi_model.sample_posterior(
        model=resolvi_model.module.model_residuals,
        return_sites=["mixture_proportions"],
        summary_fun={"post_sample_means": np.mean},
        num_samples=10,
        summary_frequency=100
    )
    samples = pd.DataFrame(samples).T

    # Assign proportions and generated expression
    adata.obs[["true_proportion", "diffusion_proportion", "background_proportion"]] = samples.loc[
        "post_sample_means", "mixture_proportions"
    ]
    adata.layers["generated_expression"] = samples_corr.loc["post_sample_q50", "px_rate"]

In [44]:
for ds in datasets:
    counts_df, metadata = load_data(os.path.join(MERSCOPE_DATADIR, 'cell_meta_subset_tx_reassigned.csv'), os.path.join(MERSCOPE_DATADIR, f'synthetic_{ds}_transcript_meta.csv'))
    adata = create_adata_object(counts_df, metadata)

    start_memory = memory_usage()
    start_time = time.time()

    resolvi_model = setup_resolvi(adata, accelerator='cpu')

    resolvi_predict(adata, resolvi_model)

    get_latent_representation_and_neighbors(adata, resolvi_model)

    sample_posterior_and_add_to_adata(adata, resolvi_model)

    end_time = time.time()
    end_memory = memory_usage()
    
    execution_time = end_time - start_time
    memory_diff = end_memory - start_memory
    adata.uns['execution_time'] = execution_time
    adata.uns['memory_used'] = memory_diff

    adata.write_h5ad(os.path.join(RESOLVI_RESULTS_DIR, f'synthetic_{ds}_supervised_resolvi.h5ad'))

/var/folders/h6/mz7d052n7mv3whgg0g3ck4f40000gn/T/ipykernel_68198/1465257313.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  transcript_metadata = pd.read_csv(tx_metadata_filepath).rename({'Unnamed: 0': 'molecule_id'}, axis=1)


INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        


/Users/smaffa/miniconda3/envs/resolvi/lib/python3.10/site-packages/scvi/data/fields/_dataframe_field.py:187: UserWarning: Category 0 in adata.obs['_scvi_ind_x'] has fewer than 3 cells. Models may not train properly.
  categorical_mapping = _make_column_categorical(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/smaffa/miniconda3/envs/resolvi/lib/python3.10/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/smaffa/miniconda3/envs/resolvi/lib/python3.10/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
/Users/smaffa/miniconda3/env

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
/Users/smaffa/miniconda3/envs/resolvi/lib/python3.10/site-packages/scvi/model/base/_pyromixin.py:499: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  _, _, device = parse_device_args(


...

Sampling local variables, batch:   0%|          | 0/142 [00:00<?, ?it/s]

/Users/smaffa/miniconda3/envs/resolvi/lib/python3.10/site-packages/scvi/external/resolvi/_utils.py:64: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  _, _, device = parse_device_args(
/Users/smaffa/miniconda3/envs/resolvi/lib/python3.10/site-packages/scvi/model/base/_pyromixin.py:499: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  _, _, device = parse_device_args(


Sampling local variables, batch:   0%|          | 0/553 [00:00<?, ?it/s]

/Users/smaffa/miniconda3/envs/resolvi/lib/python3.10/site-packages/scvi/model/base/_pyromixin.py:499: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  _, _, device = parse_device_args(


Sampling local variables, batch:   0%|          | 0/553 [00:00<?, ?it/s]